# Experiment 40 — Backward-free SparseWalker with hard negatives

This is a surgical extension of Experiment 39. Same corrected SparseWalker v1.1 recurrence, same local forward-only plasticity, **no warm start, no optimizer, no backward, no autograd gradients**.

Epochs 1–5 use the original random-negative rule. From epoch 6 onward each local contrastive update uses **8 hard negatives mined from the model's own current scores + 8 popularity negatives + 16 random negatives**.

The primary bar is the previous backward-free best validation NDCG@10: **0.040516**. SASRec validation is **0.042968**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, runpy, json, torch
from pathlib import Path

REPO='/content/Sparsewalker'
BRANCH='agent/local-hard-negative-walker-v2'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO], check=True)
for p in [f'{REPO}/src', f'{REPO}/experiments', f'{REPO}/benchmarks']:
    if p not in sys.path: sys.path.insert(0,p)
import sparsewalker
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__)
print('BRANCH',BRANCH,'PACKAGE',sparsewalker.__file__)


## Fresh run

Set `RUN_FRESH=True` for a new from-scratch run. `best.pt` and `last.pt` are written to Drive every epoch, so a disconnect is recoverable.


In [ ]:
RUN_FRESH=True
if RUN_FRESH:
    SCRIPT=f'{REPO}/experiments/run_amazon_local_hard_negative_walker.py'
    sys.argv=[SCRIPT,
        '--dataset','beauty',
        '--epochs','70',
        '--batch-size','512',
        '--hard-warmup-epochs','5',
        '--hard-negatives','8',
        '--popularity-negatives','8',
        '--total-negatives','32',
        '--hard-pool-size','512',
    ]
    runpy.run_path(SCRIPT, run_name='__main__')


## Resume after a Colab crash

Run setup above, set `RUN_FRESH=False`, then execute this cell. It loads `last.pt` and continues from the following epoch.


In [ ]:
RESUME=False
if RESUME:
    SCRIPT=f'{REPO}/experiments/run_amazon_local_hard_negative_walker.py'
    sys.argv=[SCRIPT,'--dataset','beauty','--epochs','70','--batch-size','512','--resume']
    runpy.run_path(SCRIPT, run_name='__main__')


## Test the saved best checkpoint only

This requires no training. Useful after a crash once `best.pt` exists.


In [ ]:
TEST_ONLY=False
if TEST_ONLY:
    SCRIPT=f'{REPO}/experiments/run_amazon_local_hard_negative_walker.py'
    sys.argv=[SCRIPT,'--dataset','beauty','--test-only']
    runpy.run_path(SCRIPT, run_name='__main__')


## Inspect trajectory


In [ ]:
import pandas as pd
root=Path('/content/drive/MyDrive/sparsewalker_local_hard_negative/beauty/seed42')
hp=root/'history.json'
if hp.exists():
    df=pd.DataFrame(json.loads(hp.read_text()))
    cols=['epoch','negative_mode','mean_contrastive_margin','mean_positive_prob','mean_negative_prob','mean_hard_negative_score','mean_context_error','val_NDCG@10','val_HR@10','positions_per_s']
    display(df[[c for c in cols if c in df.columns]])
    if len(df):
        best=df.loc[df['val_NDCG@10'].idxmax()]
        print('BEST',best.to_dict())
        print('PREVIOUS_LOCAL_CONTRASTIVE_BEST',0.040515991131704246)
        print('SASREC_VAL',0.04296780165590764)
else:
    print('No history yet')


### What to watch

- Epochs 1–5 should resemble Experiment 39; this is a built-in sanity control.
- At epoch 6, `negative_mode` switches to `mixed_hard`. Random-negative probability may rise because the task becomes harder; that is expected.
- The key question is whether NDCG rises above **0.040516** and ideally crosses SASRec validation **0.042968**.
- If hard-negative score rises while NDCG falls, the new mining rule is overfitting the model's own mistakes rather than improving ranking.
